# Structure Complète du Projet

projet-ebola-prediction/
│
├── 📁 .streamlit/
│   └── config.toml
│
├── 📁 data/
│   └── donnees_epidemiologiques_RDC.csv
│
├── 📁 models/
│   ├── model.pkl
│   └── scaler.pkl
│
├── 📁 notebooks/
│   └── Epi_reg_complet.ipynb
│
├── 📁 utils/
│   └── data_processing.py
│
├── app.py
├── train_model.py
├── requirements.txt
├── README.md
├── .gitignore
└── Dockerfile (optionnel)

# Fichiers Détailés
## 1. .streamlit/config.toml
Configuration de l'interface Streamlit.

[theme]
primaryColor = "#2E86C1"
backgroundColor = "#F8F9FA"
secondaryBackgroundColor = "#E5E8E8"
textColor = "#212529"
font = "sans serif"

[server]
maxUploadSize = 10
enableXsrfProtection = true

[browser]
gatherUsageStats = false

# 2. data/donnees_epidemiologiques_RDC.csv
Votre fichier de données (à placer ici). Assurez-vous qu'il est bien nommé.

# 3.  models/
Dossier où seront sauvegardés les modèles entraînés.



# 4. notebooks/Epi_reg_complet.ipynb
Votre notebook avec les modifications pour sauvegarder le modèle.

In [ ]:
import joblib
import os

# Créer le dossier models s'il n'existe pas
os.makedirs('models', exist_ok=True)

# Sauvegarde du modèle et du scaler
joblib.dump(model, 'models/model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
print(" Modèle et scaler sauvegardés dans le dossier 'models/'")

# 5. utils/data_processing.py
Fonctions utilitaires pour le traitement des données.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def load_and_prepare_data(filepath):
    """
    Chargement et préparation des données pour l'entraînement.
    
    Args:
        filepath (str): Chemin vers le fichier CSV
        
    Returns:
        tuple: (X, y, feature_names)
    """
    df = pd.read_csv(filepath)
    
    # Sélection des features et de la cible
    feature_columns = ['Cas_Suspects', 'Pluviometrie_mm', 'Temperature_C', 
                      'Deforestation_ha', 'NDVI']
    target_column = 'Cas_Confirmes'
    
    X = df[feature_columns]
    y = df[target_column]
    
    return X, y, feature_columns

def create_input_dataframe(cas_suspects, pluviometrie, temperature, deforestation, ndvi):
    """
    Création du DataFrame à partir des entrées utilisateur.
    
    Args:
        cas_suspects (int): Nombre de cas suspects
        pluviometrie (float): Pluviométrie en mm
        temperature (float): Température en °C
        deforestation (float): Déforestation en ha
        ndvi (float): Indice NDVI
        
    Returns:
        pd.DataFrame: DataFrame formaté pour la prédiction
    """
    return pd.DataFrame({
        'Cas_Suspects': [cas_suspects],
        'Pluviometrie_mm': [pluviometrie],
        'Temperature_C': [temperature],
        'Deforestation_ha': [deforestation],
        'NDVI': [ndvi]
    })

def validate_inputs(cas_suspects, pluviometrie, temperature, deforestation, ndvi):
    """
    Valide les entrées utilisateur.
    
    Returns:
        tuple: (is_valid, error_message)
    """
    if cas_suspects < 0:
        return False, "Les cas suspects ne peuvent pas être négatifs"
    if pluviometrie < 0:
        return False, "La pluviométrie ne peut pas être négative"
    if temperature < -20 or temperature > 50:
        return False, "La température doit être entre -20°C et 50°C"
    if deforestation < 0:
        return False, "La déforestation ne peut pas être négative"
    if ndvi < 0 or ndvi > 1:
        return False, "Le NDVI doit être entre 0 et 1"
    return True, "OK"

# 6. train_model.py
Script pour entraîner et sauvegarder le modèle.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from utils.data_processing import load_and_prepare_data

def train_model():
    """
    Entraîne le modèle de régression linéaire multiple.
    """
    print(" Début de l'entraînement du modèle...")
    
    # 1. Chargement des données
    X, y, feature_names = load_and_prepare_data('data/donnees_epidemiologiques_RDC.csv')
    
    print(f" Données chargées : {X.shape[0]} observations, {X.shape[1]} variables")
    
    # 2. Division train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    print(f" Train set: {X_train.shape[0]} observations")
    print(f" Test set: {X_test.shape[0]} observations")
    
    # 3. Standardisation
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 4. Entraînement
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    
    # 5. Évaluation
    y_pred = model.predict(X_test_scaled)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print("\n Performance du modèle :")
    print(f"   MSE  : {mse:.4f}")
    print(f"   RMSE : {rmse:.4f}")
    print(f"   MAE  : {mae:.4f}")
    print(f"   R²   : {r2:.4f}")
    
    # 6. Affichage des coefficients
    print("\n Coefficients du modèle :")
    for name, coef in zip(feature_names, model.coef_):
        print(f"   {name}: {coef:.4f}")
    print(f"   Intercept: {model.intercept_:.4f}")
    
    # 7. Sauvegarde
    os.makedirs('models', exist_ok=True)
    joblib.dump(model, 'models/model.pkl')
    joblib.dump(scaler, 'models/scaler.pkl')
    
    print("\n Modèle sauvegardé dans 'models/model.pkl'")
    print(" Scaler sauvegardé dans 'models/scaler.pkl'")
    
    return model, scaler

if __name__ == "__main__":
    train_model()

# 7.  app.py
Application Streamlit principale.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
from utils.data_processing import validate_inputs, create_input_dataframe

# Configuration de la page
st.set_page_config(
    page_title="Prédiction Ebola - RDC",
    page_icon=" ",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Style CSS personnalisé
st.markdown("""
<style>
    .main-header {
        font-size: 2.5rem;
        color: #2E86C1;
        text-align: center;
        margin-bottom: 1rem;
    }
    .prediction-box {
        background-color: #D4E6F1;
        padding: 1.5rem;
        border-radius: 10px;
        text-align: center;
        margin: 1rem 0;
    }
    .prediction-number {
        font-size: 3rem;
        font-weight: bold;
        color: #1A5276;
    }
    .info-box {
        background-color: #FDEBD0;
        padding: 1rem;
        border-radius: 10px;
        border-left: 5px solid #E67E22;
    }
    .stButton > button {
        width: 100%;
        background-color: #2E86C1;
        color: white;
        font-weight: bold;
    }
    .stButton > button:hover {
        background-color: #1A5276;
    }
</style>
""", unsafe_allow_html=True)

# Chargement du modèle
@st.cache_resource
def load_model():
    """Charge le modèle et le scaler."""
    try:
        model = joblib.load('models/model.pkl')
        scaler = joblib.load('models/scaler.pkl')
        return model, scaler
    except FileNotFoundError:
        st.error(" Modèle non trouvé. Veuillez exécuter train_model.py d'abord.")
        st.stop()
    except Exception as e:
        st.error(f" Erreur lors du chargement du modèle : {e}")
        st.stop()

model, scaler = load_model()

# Sidebar
with st.sidebar:
    st.image("https://cdn-icons-png.flaticon.com/512/190/190411.png", width=100)
    st.markdown("## Prédiction Ebola")
    st.markdown("---")
    st.markdown("### À propos")
    st.markdown("""
    Cette application utilise un modèle de **régression linéaire multiple** 
    pour prédire le nombre de cas confirmés d'Ebola en RDC.
    
    **Variables utilisées :**
    - Cas suspects
    - Pluviométrie
    - Température
    - Déforestation
    - NDVI
    """)
    st.markdown("---")
    st.caption(" Projet Data Science - UMIE/DRC")

# En-tête principal
st.markdown('<p class="main-header"> Prédiction des cas confirmés d\'Ebola</p>', unsafe_allow_html=True)

# Colonnes pour la mise en page
col_info, col_form = st.columns([1, 2])

with col_info:
    st.markdown("""
    ###  Comment ça fonctionne ?
    
    1. Saisissez les données dans le formulaire
    2. Cliquez sur **Prédire**
    3. Obtenez instantanément le nombre de cas confirmés prédits
    
    ###  Performance du modèle
    """)
    
    # Afficher les métriques du modèle
    try:
        # Ces valeurs devraient être chargées depuis un fichier
        st.metric("R²", "0.94", "Excellent")
        st.metric("RMSE", "5.50", "Faible")
        st.metric("MAE", "4.42", "Faible")
    except:
        pass

with col_form:
    with st.form("prediction_form"):
        st.markdown("### Saisie des données")
        
        col1, col2 = st.columns(2)
        
        with col1:
            cas_suspects = st.number_input(
                " Cas suspects",
                min_value=0,
                value=70,
                step=5,
                help="Nombre de cas suspects signalés"
            )
            
            pluviometrie = st.number_input(
                " Pluviométrie (mm)",
                min_value=0,
                value=165,
                step=10,
                help="Précipitations en millimètres"
            )
            
            temperature = st.number_input(
                " Température (°C)",
                min_value=10.0,
                max_value=40.0,
                value=25.8,
                step=0.1,
                help="Température moyenne en degrés Celsius"
            )
        
        with col2:
            deforestation = st.number_input(
                " Déforestation (ha)",
                min_value=0.0,
                value=17.0,
                step=0.5,
                help="Surface déforestée en hectares"
            )
            
            ndvi = st.slider(
                " NDVI",
                min_value=0.0,
                max_value=1.0,
                value=0.44,
                step=0.01,
                help="Indice de végétation (0 = sol nu, 1 = végétation dense)"
            )
        
        st.markdown("---")
        submitted = st.form_submit_button(" Prédire les cas confirmés", type="primary")

# Traitement de la prédiction
if submitted:
    # Validation des entrées
    is_valid, error_msg = validate_inputs(cas_suspects, pluviometrie, temperature, deforestation, ndvi)
    
    if not is_valid:
        st.error(f" Erreur de saisie : {error_msg}")
    else:
        # Création du DataFrame d'entrée
        input_data = create_input_dataframe(
            cas_suspects, pluviometrie, temperature, deforestation, ndvi
        )
        
        # Standardisation et prédiction
        input_scaled = scaler.transform(input_data)
        prediction = model.predict(input_scaled)[0]
        prediction = max(0, round(prediction, 1))
        
        # Affichage des résultats
        st.divider()
        
        # Boîte de résultat
        st.markdown(f"""
        <div class="prediction-box">
            <p style="font-size: 1.2rem; margin: 0;"> Nombre de cas confirmés prédits</p>
            <p class="prediction-number">{prediction}</p>
        </div>
        """, unsafe_allow_html=True)
        
        # Colonnes pour les informations complémentaires
        col_res1, col_res2, col_res3 = st.columns(3)
        
        with col_res1:
            st.metric(" Cas suspects", cas_suspects)
        with col_res2:
            st.metric(" Pluviométrie", f"{pluviometrie} mm")
        with col_res3:
            st.metric(" Température", f"{temperature} °C")
        
        # Graphique des variables
        with st.expander(" Visualisation des données d'entrée", expanded=False):
            fig = px.bar(
                x=['Cas Suspects', 'Pluviométrie', 'Température', 'Déforestation', 'NDVI'],
                y=[cas_suspects, pluviometrie, temperature, deforestation, ndvi],
                labels={'x': 'Variables', 'y': 'Valeurs'},
                title="Distribution des variables d'entrée",
                color=['#2E86C1', '#28B463', '#E67E22', '#7D3C98', '#F1C40F']
            )
            fig.update_layout(showlegend=False)
            st.plotly_chart(fig, use_container_width=True)
        
        # Interprétation
        with st.expander(" Interprétation des résultats", expanded=True):
            st.markdown("""
            <div class="info-box">
                <b> Analyse :</b><br>
                • Le nombre de <b>cas suspects</b> est le facteur le plus influent<br>
                • La <b>pluviométrie</b> et la <b>déforestation</b> ont un impact positif modéré<br>
                • Une <b>température</b> plus élevée et un <b>NDVI</b> plus élevé réduisent la prédiction<br>
                • Le modèle explique environ <b>94%</b> de la variabilité des cas confirmés
            </div>
            """, unsafe_allow_html=True)

# Pied de page
st.divider()
col_footer1, col_footer2 = st.columns([2, 1])
with col_footer1:
    st.caption(" Développé dans le cadre du projet de Data Science - UMIE/DRC")
with col_footer2:
    st.caption("[ Voir le code source](https://github.com/votre-username/projet-ebola)")

# 8. requirements.txt
Dépendances Python.

In [ ]:
streamlit>=1.28.0
pandas>=1.5.0
numpy>=1.24.0
scikit-learn>=1.2.0
joblib>=1.2.0
plotly>=5.17.0
matplotlib>=3.6.0
seaborn>=0.12.0

# 9. README.md
Documentation du projet.

# Prédiction des cas confirmés d'Ebola en RDC

##  Description
Application de prédiction des cas confirmés d'une epidemie basée sur un modèle de régression linéaire multiple utilisant des données épidémiologiques et environnementales.

##  Fonctionnalités
- Prédiction en temps réel des cas confirmés
- Interface utilisateur intuitive avec Streamlit
- Visualisation des données d'entrée
- Interprétation des résultats

## Variables utilisées
| Variable | Description |
|----------|-------------|
| Cas suspects | Nombre de cas suspects signalés |
| Pluviométrie | Précipitations en mm |
| Température | Température moyenne en °C |
| Déforestation | Surface déforestée en ha |
| NDVI | Indice de végétation |

## Installation

### Prérequis
- Python 3.8 ou supérieur
- Git

### Étapes d'installation
```bash
# 1. Cloner le dépôt
git clone https://github.com/Onesime243/projet-Epi_INHOHA.git 
cd Epi_regression

# 2. Créer un environnement virtuel
python -m venv venv
source venv/bin/activate  # Sur Windows: venv\Scripts\activate

# 3. Installer les dépendances
pip install -r requirements.txt

# 4. Entraîner le modèle
python train_model.py

# 5. Lancer l'application
streamlit run app.py

# Guide Complet de Déploiement sur GitHub et Streamlit Cloud
## Prérequis

Avant de commencer, assurez-vous d'avoir :

- Un compte GitHub

- Un compte Streamlit Cloud




 # Créer le dépôt sur GitHub

1. Allez sur GitHub

2. Cliquez sur le bouton New ou + en haut à droite

3. Remplissez les informations :

 - Repository name : projet-ebola

 - Description : "Application de prédiction des cas confirmés d'Ebola en RDC"

 - Public ou Private (choisissez Public pour que tout le monde puisse le voir)

 - Ne pas cocher "Initialize this repository with a README"

Cliquez sur Create repository

# 4.2 Connecter votre dépôt local à GitHub

In [ ]:
# Ajouter l'URL du dépôt distant
git remote add origin https://github.com/votre-username/projet-ebola.git

# Vérifier la connexion
git remote -v

# Pousser le code vers GitHub
git push -u origin main

# Étape 5 : Déploiement sur Streamlit Cloud

# Étapes Détaillées
###  Étape 1 : Accéder à Streamlit Cloud

Allez sur share.streamlit.io 

Connectez-vous avec votre compte GitHub

### Étape 2 : Cliquer sur "New app"

### Étape 3 : Remplir les informations
Dans le formulaire qui s'ouvre, vous verrez :

┌─────────────────────────────────────────────────────────┐
│    Repository                                         │
│  ┌───────────────────────────────────────────────────┐ │
│  │  Onesime243/projet-Epi_INHOHA                    │ │
│  └───────────────────────────────────────────────────┘ │
│                                                         │
│    Branch                                             │
│  ┌───────────────────────────────────────────────────┐ │
│  │  main                                             │ │
│  └───────────────────────────────────────────────────┘ │
│                                                         │
│  📄 Main file path                                     │
│  ┌───────────────────────────────────────────────────┐ │
│  │  app.py                     ←   C'est ICI !     │ │
│  └───────────────────────────────────────────────────┘ │
│                                                         │
│    [Deploy]                                            │
└─────────────────────────────────────────────────────────┘

### Étape 4 : Saisir le chemin

Dans le champ Main file path, entrez :

app.py


 **Important** : Ce doit être le nom exact de votre fichier principal.